In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer

/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODELTYPE = "deepset/gbert-base"
DATASET = "med_indication_all_RF_diag"
DATASETPATH =  Path("../data") / f"ind.{DATASET}"
DATASETFILE = Path("../data") / "medindcls_bert.json"

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

tokenizer = AutoTokenizer.from_pretrained(MODELTYPE)

if not DATASETFILE.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, clean=False)
    with open(DATASETFILE, "wb") as f:
        print(f"Saving dataset under {DATASETFILE}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {DATASETFILE}")
    with open(DATASETFILE, "rb") as f:
        dataset = pickle.load(f)

27996
Loading dataset from: ../data/medindcls_bert.json


/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator OneHotEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/tmp/ipykernel_682214/3214339880.py:21: DeprecationWarning: Please use `csr_matrix` from the `scipy.sparse` namespace, the `scipy.sparse.csr` namespace is deprecated.
  dataset = pickle.loa

In [4]:
mixfactor = 0.5
ig_gcn_only_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_only.json"
shap_gcn_only_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_only.json"
ig_gcn_bert_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_bert.json"
shap_gcn_bert_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_bert.json"

ig_gcn_only_values = pickle.load(open(ig_gcn_only_path, "rb"))
ig_gcn_bert_values = pickle.load(open(ig_gcn_bert_path, "rb"))
shap_gcn_only_values = pickle.load(open(shap_gcn_only_path, "rb"))
shap_gcn_bert_values = pickle.load(open(shap_gcn_bert_path, "rb"))

In [5]:
np.array(ig_gcn_only_values).shape

(540, 2699)

In [6]:
top_n_interpret = 10
top_ig_gcn_only_values = np.argpartition(ig_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_only_values = np.argpartition(shap_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_bert_values = np.argpartition(shap_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_only_values.max(), top_ig_gcn_only_values.shape

(2698, (540, 10))

In [7]:
random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)


def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [8]:
top_ig_gcn_only_values = np.vectorize(map_to_idx)(top_ig_gcn_only_values)
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)
top_shap_gcn_only_values = np.vectorize(map_to_idx)(top_shap_gcn_only_values)
top_shap_gcn_bert_values = np.vectorize(map_to_idx)(top_shap_gcn_bert_values)

In [9]:
first_order_adj = adj @ adj
first_order_adj = first_order_adj.toarray()

In [22]:
top_n_input = 10
top_input_test_rel_nodes = np.argpartition(first_order_adj[test_mask][:, doc_mask], -top_n_input)[:, -top_n_input:]
top_input_test_rel_nodes = np.vectorize(map_to_idx)(top_input_test_rel_nodes)
top_input_test_rel_nodes.max(), top_input_test_rel_nodes.shape

(2688, (540, 10))

In [23]:
s = 0
for a, b in zip(top_input_test_rel_nodes, top_ig_gcn_only_values):
    s += len(np.intersect1d(a, b))
print(s)

s = 0
for a, b in zip(top_input_test_rel_nodes, top_ig_gcn_bert_values):
    s += len(np.intersect1d(a, b))
print(s)

s = 0
for a, b in zip(top_input_test_rel_nodes, top_shap_gcn_only_values):
    s += len(np.intersect1d(a, b))
print(s)

s = 0
for a, b in zip(top_input_test_rel_nodes, top_shap_gcn_bert_values):
    s += len(np.intersect1d(a, b))
print(s)

2590
2495
2471
2480


In [24]:
s = list()
for a, b in zip(top_input_test_rel_nodes, top_ig_gcn_only_values):
    s.append(np.setdiff1d(b, a))
intersect_df = pd.DataFrame({"nodes": s}, index=test_idx)
intersect_df

,nodes
2298,"[2282, 2284, 2286]"
2130,"[818, 819, 820, 821]"
2559,"[580, 582, 587, 589]"
1046,"[122, 123, 124, 125, 1378, 1379, 1380, 2163, 2..."
1991,"[1254, 1351, 1354, 1355, 1356, 1357, 1718, 1719]"
...,...
2094,"[291, 292, 293, 294, 1334, 1335, 1337, 1441, 2..."
1060,"[1466, 1474]"
165,"[619, 620, 675, 676, 1912, 2149, 2551]"
1722,"[1728, 1732, 1734]"
